# Running SHiP #

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# import redpandda
# from redpandda import *
# import geostas

# import visualizations
# import comodo
import hdbscan
import time

from SHiP import SHiP
from SHiP.ultrametric_tree import UltrametricTreeType as UTreeType, AVAILABLE_ULTRAMETRIC_TREE_TYPES
from SHiP.partitioning import PartitioningMethod as PMethod, AVAILABLE_PARTITIONING_METHODS
# from redpandda import preprocessing, preprocess_protein_trajectory
import redpandda_general
from clustering_functions import clustering_workflow
from compare_clusterings import *
import clustering_functions
# from timestep_clustering import *
import distance_matrix as dm
import seaborn as sns
import compare_clusterings as cc
# import resicon
# from resicon import *

# import geostas
# import mdtraj as md
# import MDAnalysis as mda 
# from redpandda import *

In [8]:
def preprocess_protein_trajectory(prot_info, k_cluster=None):
  frames_count = prot_info[3]
  traj_array, k_cluster = preprocessing(prot_info,frames_count,k_cluster)
  return traj_array, k_cluster

load molecular dynamcis dataset:

must consist of the following files:
* xtc or dcd trajectory file
* pdb peptide file
* folder where first two files are stored
* frame count (optional, otherwise None)
* k clusters (optional, otherwise None)

In [11]:
# md_trajectory_info = ['trajectory-2.xtc','fs-peptide.pdb','McGibbon/',None,None]
md_trajectory_info = ['prod_r1_nojump_prot.xtc','prod_r1_pbc_fit_prot_last.pdb','proteins_comet/ProtNo2/',None,None]


# md_trajectory_info = ['1hhp.dcd','1hhp.pdb','HIV1Protease/',None,None]

#how many frames to process
frames_count = md_trajectory_info[3]

trajectory_file = md_trajectory_info[0].split()[0]
pdb_file = md_trajectory_info[1].split()[0]

# call MD-related preprocessing
traj_array, k_cluster = preprocess_protein_trajectory(md_trajectory_info)
np.savez("data/traj_data_tll.npz", traj_array=traj_array, k_cluster=k_cluster)

In [5]:
import numpy as np
filename = "traj_data_tll"
data = np.load("data/"+filename+".npz",)

traj_array = data["traj_array"]
k_cluster = int(data["k_cluster"])  # convert back to int if needed

print(traj_array.shape)


(40001, 269, 3)


In [ ]:
import os
import pandas as pd
from clustering_functions import clustering_workflow
from SHiP.partitioning import PartitioningMethod as PMethod
import redpandda_general as rp  # if needed for trajectory preparation
import compare_clusterings as cc  # for Q computation

# Example input

matrices_to_apply = ["delta+1std"]

clusterings_to_apply = [
    {
        "name": "SHiP_1",
        "method": "ship",
        "params": {
            "partitioning_method": PMethod.Elbow,
            "hierarchie": 2,
            "tiebreaker_method": "euclidean_distance"
        }
    }
]

# ---- Collect results in a list ---- #
results_list = []

print(f"\n🚀 Processing {filename} with SHiP...")

# Run clustering workflow
res_list = clustering_workflow(
    traj_array,
    matrices_to_apply,
    clusterings_to_apply,
    post_process_noise=True
)

# Loop over results returned by workflow
for r in res_list:
    # Compute Q
    dist_matrices = rp.get_distance_matrices(traj_array)
    curr_clustering = r["clustering"]
    try:
        Q, _ = cc.get_Q_for_clustering(dist_matrices, curr_clustering, k_cluster)
    except Exception as e:
        print("⚠️ Error computing Q:", e)
        Q = None

    # Append result
    results_list.append({
        "dataset": filename,
        "algorithm": r["name"],
        "runtime": r["runtime"],
        "Q": Q
    })

# Convert to DataFrame
df_results = pd.DataFrame(results_list)

# Save to CSV
output_csv = "data/ship_results_tll.csv"
df_results.to_csv(output_csv, index=False)

print(f"\n✅ Results saved to {output_csv}")
print(df_results)


🚀 Processing traj_data_tll with SHiP...
avg delta (269, 269)
compute_mutual_reachability_dists_from_dmat() called with 2
1.5816
0.1527
Construct hierarchy from distance matrix:
DCTree from distance matrix built!
[WARNING] Cost of a parent node is smaller than its child. Try automatic fixing...
annotate_tree for distance matrix: start optimization...
SHiP distance matrix constructor finished!
Compute tree from distance matrix...
annotate_tree for distance matrix: start optimization...
compute_mutual_reachability_dists_from_dmat() called with 2
Construct hierarchy from distance matrix:
DCTree from distance matrix built!
[WARNING] Cost of a parent node is smaller than its child. Try automatic fixing...
annotate_tree for distance matrix: start optimization...
SHiP distance matrix constructor finished!
Compute tree from distance matrix...
annotate_tree for distance matrix: start optimization...

✅ Results saved to ship_results.csv
         dataset algorithm     runtime         Q
0  traj_da